In [3]:
import os
import subprocess
import sys

print("--- Diagnostics ---")
print(f"1. Current User: {os.getlogin() if hasattr(os, 'getlogin') else 'unknown'}")
print(f"2. JAVA_HOME:    {os.environ.get('JAVA_HOME', '❌ NOT SET')}")

try:
    java_path = subprocess.check_output(['which', 'java']).decode().strip()
    print(f"3. Java Path:    {java_path}")
    java_ver = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT).decode().strip()
    print(f"4. Java Version:\n{java_ver}")
except Exception as e:
    print(f"❌ Java check failed: {e}")

print(f"5. Python Exec:  {sys.executable}")

--- Diagnostics ---
1. Current User: massimiliano
2. JAVA_HOME:    /usr/lib/jvm/java-21-openjdk
3. Java Path:    /usr/lib/jvm/java-21-openjdk/bin/java
4. Java Version:
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7)
OpenJDK 64-Bit Server VM (build 21.0.10+7, mixed mode, sharing)
5. Python Exec:  /home/massimiliano/Work/DistArch/Spark/.venv/bin/python


In [ ]:
from pyspark.sql import SparkSession

# Create session
spark = SparkSession.builder \
    .appName("templateApp") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print(f"✅ SUCCESS!")
print(f"Spark: {spark.version}")
print(f"Java:  {spark._jvm.java.lang.System.getProperty('java.version')}")

## Load a CSV file

In [ ]:
# Option 1: load as a Spark DataFrame using read.format()

# Load the CSV file
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("path/to/your_file.csv") # Suppose root where you called `uv run jupyter lab` was DistrArch/

# Show the results
df.show()

In [ ]:
# Option 2: load as a Spark DataFrame using read.csv()

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define the structure
manual_schema = StructType([
    StructField("User_ID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("City", StringType(), True)
])

# Load with the schema
df = spark.read.csv("path/to/file.csv", header=True, schema=manual_schema) # Suppose root where you called `uv run jupyter lab` was DistrArch/

In [ ]:
# Option 3: load using RDDs

from pyspark import SparkContext, SparkConf

# Initialize SparkContext
conf = SparkConf().setAppName("RDD_CSV_Example").setMaster("local[*]")
sc = SparkContext(conf=conf)

# Load the text file as an RDD
# textFile reads every line as a single string
raw_rdd = sc.textFile("path/to/your_file.csv") # Suppose root where you called `uv run jupyter lab` was DistrArch/

# Handle the CSV logic manually
# Extract the header so we can filter it out
header = raw_rdd.first() 

# Parse the lines: filter the header, split by comma, and map to a list/tuple
parsed_rdd = raw_rdd \
    .filter(lambda line: line != header) \
    .map(lambda line: line.split(","))

# Perform an action (e.g., take the first 5 rows)
results = parsed_rdd.take(5)

for row in results:
    print(row)

# Stop the context
sc.stop()